### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

This notebook will:
1. Delete all resources and purge soft-deleted services (APIM, Key Vault, Cognitive Services)
2. Delete the resource group
3. Sweep for any remaining soft-deleted resources that might block future redeployments

In [ ]:
import os, sys
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group = "azureml-model-lab"

# This deletes all resources, then purges soft-deleted APIM, Key Vault, and Cognitive Services
utils.cleanup_resources(deployment_name, resource_group_name=resource_group)

### 🧹 Purge any remaining soft-deleted resources

If the resource group was already deleted (e.g., via the Azure Portal), soft-deleted APIM, Key Vault, and Cognitive Services resources may still linger and block redeployment with the same names. This cell sweeps for and purges them.

In [ ]:
# Clear stale deployment records (prevents ETag/IfMatch errors on re-deploy)
dep_list = utils.run(f'az deployment group list -g {resource_group} --query "[].name" -o json', "", "", print_output=False)
if dep_list.success and dep_list.json_data:
    for dep_name in dep_list.json_data:
        utils.run(f'az deployment group delete -g {resource_group} -n {dep_name}', "", "", print_output=False)
    utils.print_info(f"Cleared {len(dep_list.json_data)} stale deployment record(s)")

# Purge soft-deleted Cognitive Services belonging to this resource group
cs_output = utils.run('az cognitiveservices account list-deleted', "", "", print_output=False)
if cs_output.success and cs_output.json_data:
    for deleted in cs_output.json_data:
        loc = deleted.get("location", "")
        name = deleted.get("name", "") or deleted.get("properties", {}).get("resourceName", "")
        rg = deleted.get("properties", {}).get("resourceGroup", "")
        if rg.lower() == resource_group.lower() and name:
            utils.print_info(f"Purging soft-deleted Cognitive Services: {name}")
            utils.run(f'az cognitiveservices account purge -n {name} -g {resource_group} -l "{loc}"',
                f"Purged {name}", f"Failed to purge {name} (may already be purged)")

# Purge soft-deleted APIM services belonging to this resource group
apim_output = utils.run('az apim deletedservice list', "", "", print_output=False)
if apim_output.success and apim_output.json_data:
    for deleted in apim_output.json_data:
        svc_id = deleted.get("serviceId", "")
        name = deleted.get("name", "")
        loc = deleted.get("location", "")
        if f"/resourceGroups/{resource_group}/" in svc_id and name:
            utils.print_info(f"Purging soft-deleted APIM: {name}")
            utils.run(f'az apim deletedservice purge --service-name {name} --location "{loc}"',
                f"Purged {name}", f"Failed to purge {name}")

# Purge soft-deleted Key Vaults belonging to this resource group
kv_output = utils.run('az keyvault list-deleted', "", "", print_output=False)
if kv_output.success and kv_output.json_data:
    for deleted in kv_output.json_data:
        rg_match = deleted.get("properties", {}).get("vaultId", "")
        name = deleted.get("name", "")
        loc = deleted.get("properties", {}).get("location", "")
        if f"/resourceGroups/{resource_group}/" in rg_match and name:
            utils.print_info(f"Purging soft-deleted Key Vault: {name}")
            utils.run(f'az keyvault purge --name {name} --location "{loc}"',
                f"Purged {name}", f"Failed to purge {name}")

# Purge soft-deleted ML workspaces belonging to this resource group
# Use REST API with forceToPurge=true since az ml workspace purge is unreliable
sub_output = utils.run('az account show --query id -o tsv', "", "", print_output=False)
if sub_output.success:
    sub_id = sub_output.text.strip()
    ml_output = utils.run(
        f'az rest --method GET --url "/subscriptions/{sub_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces?api-version=2024-04-01&includeDeletedWorkSpaces=true" --query "value[?properties.provisioningState==\'SoftDeleted\'].name" -o json',
        "", "", print_output=False
    )
    if ml_output.success and ml_output.json_data:
        for name in ml_output.json_data:
            utils.print_info(f"Purging soft-deleted ML Workspace: {name}")
            utils.run(
                f'az rest --method DELETE --url "/subscriptions/{sub_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces/{name}?api-version=2024-04-01&forceToPurge=true"',
                f"Purged {name}", f"Failed to purge {name}"
            )

utils.print_ok("Soft-deleted resource purge complete")